# Bertje Sentence Analysis

## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
# read the sentence data
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/input_data/translated_sentence_data.xlsx")
# check the head
sentence_df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,0,I was rushed into the [PRESSION-1] [LOCATION-1...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",1,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,2,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",3,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",4,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_datasets/stopwords/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg"
]

dutch_stopwords.extend(extra_list)

### 1.3 Embed sentences with BERTje

In [5]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import numpy as np
# load saved bertje sentence-embeddings # we have two different embeddings due to different calculations 
embeddings = np.load("../../data/saved_embeddings/sentence_embeddings_bertje_nl.npy")
#or
#embeddings = np.load("../../data/saved_embeddings/sentence_embeddings_bertje_nl_new.npy") 

In [ ]:
# pre-calculate the sentence embeddings with BERTje if there isn't anything saved 
# calculation 1
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/saved_embeddings/sentence_embeddings_bertje_nl.npy', sentence_embeddings_nl)

In [ ]:
# calculation two

from tqdm import tqdm 
# Initialize an empty list to store embeddings
sentence_embeddings_nl = []

# Use tqdm to track progress
for sentence in tqdm(sentences, desc="Processing Embeddings", unit="sentence"):
    # Compute embedding for the sentence
    embedding = embedding_model(sentence)
    # Average token embeddings to create a sentence-level embedding
    sentence_embedding = np.mean(embedding[0], axis=0)
    # Append to the list
    sentence_embeddings_nl.append(sentence_embedding)

# Convert the list to a NumPy array
sentence_embeddings_nl = np.array(sentence_embeddings_nl)

# Check the shape of the resulting embeddings
print(f"Shape of sentence embeddings: {sentence_embeddings_nl.shape}")

# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/saved_embeddings/sentence_embeddings_bertje_nl_new.npy', sentence_embeddings_nl)

### 1.4 Pre-configure sub-models of BERTopic

In [7]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired
from umap import UMAP

# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{3,}\b')
# clustering (3 options)
hdb_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
kmeans_model = KMeans(n_clusters=100)
agglo_model = AgglomerativeClustering(n_clusters=100)
# dimension reduction
umap_model=UMAP(n_neighbors=10, n_components=2, metric='cosine', random_state=42)


In [ ]:
def build_bertopic(embedding_model, umap_model, cluster_model, vectorizer_model):
    # Initialize BERTopic with specified models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(sentences, embeddings)
    
    # Return only essential results
    return topic_model

## 2. Build/Load Model and Reduce Outliers

### 2.1 Load pre-saved model

In [220]:
from bertopic import BERTopic
# load the model (without reduced outliers)
loaded_topic_model = BERTopic.load("/workspace/mijnidbcoachnlp/data/result_data/saved_models/saved_BERTje_model", embedding_model=embedding_model)

In [148]:
loaded_topic_model.get_topic_info() # examine the topics

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18991,-1_afspraak_goed_medicatie_ontlasting,"[afspraak, goed, medicatie, ontlasting, vraag, gaan, bloed, mogelijk, krijg, krijgen]",NaN
1,0,1820,0_toilet_buikpijn_pijn_buik,"[toilet, buikpijn, pijn, buik, gevoel, last, krampen, misselijk, diarree, vaker]",NaN
2,1,694,1_sturen_wilt_recept apotheek_recept sturen,"[sturen, wilt, recept apotheek, recept sturen, apotheek sturen, nakijken, vertellen, recept, bel...",NaN
3,2,478,2_nieuw recept_nieuw_recept_graag nieuw,"[nieuw recept, nieuw, recept, graag nieuw, herhaalrecept, pentasa, nethol, puri nethol, puri, gr...",NaN
4,3,443,3_ziekenhuis_apotheek ziekenhuis_ziekenhuis apotheek_ziekenhuis recept,"[ziekenhuis, apotheek ziekenhuis, ziekenhuis apotheek, ziekenhuis recept, prikken ziekenhuis, ap...",NaN
...,...,...,...,...,...
335,334,20,334_geven buikpijn_nachts wakker_extreem_fit moe,"[geven buikpijn, nachts wakker, extreem, fit moe, darmen nachts, flinke hoofdpijn, diarree voel,...",NaN
336,335,20,335_aantal last_voet_lage rug_stijfheid pijn,"[aantal last, voet, lage rug, stijfheid pijn, steken darmen, rechtervoet, pijn stuitje, last ont...",NaN
337,336,20,336_reactie alvast_alvast bedankt_bedankt reactie_alvast,"[reactie alvast, alvast bedankt, bedankt reactie, alvast, antwoord alvast, advies alvast, bedank...",NaN
338,337,20,337_test gedaan_calprotectine test_calprotectine_calpro,"[test gedaan, calprotectine test, calprotectine, calpro, calprotectine ingeleverd, infuus huisar...",NaN


In [110]:
from bertopic import BERTopic
# load the model (without reduced outliers)
loaded_topic_model_ro = BERTopic.load("/workspace/mijnidbcoachnlp/data/result_data/saved_models/saved_BERTje_model_reduced_outliers", embedding_model=embedding_model)

In [111]:
loaded_topic_model_ro.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,240,-1_beangstigend_tijdsbestek_remmers_contactpersoon,"[beangstigend, tijdsbestek, remmers, contactpersoon, vakje, verademing, form, gescheurd, boodsch...",NaN
1,0,2231,0_toilet_pijn_buikpijn_gevoel,"[toilet, pijn, buikpijn, gevoel, buik, last, krampen, diarree, misselijk, vaker]",NaN
2,1,828,1_doorgeven_sturen_recept sturen_wilt,"[doorgeven, sturen, recept sturen, wilt, apotheek sturen, recept apotheek, vertellen, nakijken, ...",NaN
3,2,834,2_nieuw recept_nieuw_recept_graag nieuw,"[nieuw recept, nieuw, recept, graag nieuw, herhaalrecept, herhaal recept, aanvragen, nethol, her...",NaN
4,3,603,3_ziekenhuis_apotheek ziekenhuis_ziekenhuis apotheek_prikken ziekenhuis,"[ziekenhuis, apotheek ziekenhuis, ziekenhuis apotheek, prikken ziekenhuis, apotheek, ziekenhuis ...",NaN
...,...,...,...,...,...
335,334,46,334_nachts_nachts wakker_extreem_wakker,"[nachts, nachts wakker, extreem, wakker, zondagochtend, maagzuur, gewrichten voel, last maagzuur...",NaN
336,335,46,335_aantal last_voet_irritatie_gewricht,"[aantal last, voet, irritatie, gewricht, donderdagavond, enkels, warmte, heftige pijn, bepaalde ...",NaN
337,336,21,336_reactie alvast_alvast bedankt_bedankt reactie_alvast,"[reactie alvast, alvast bedankt, bedankt reactie, alvast, antwoord alvast, advies alvast, bedank...",NaN
338,337,49,337_test gedaan_calpro_test_gedaan,"[test gedaan, calpro, test, gedaan, gluten, calpro test, calprotectine thuis, abces, achtergelat...",NaN


In [23]:
#loaded_topic_model_ro.update_topics(docs=sentences, vectorizer_model=vectorizer_model) #only run if the model topic representation is not updated

In [25]:
#save again after updating topics
#loaded_topic_model_ro.save("/workspace/mijnidbcoachnlp/data/result_data/saved_models/saved_BERTje_model_reduced_outliers", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

### 2.2 Build new model

In [ ]:
from bertopic import BERTopic
# build new topic model
topic_model = build_bertopic(embedding_model=embedding_model, umap_model=umap_model, cluster_model=hdb_model, vectorizer_model=vectorizer_model)

In [ ]:
# examine the topics of the newly built model
topic_model.get_topic_info()

In [ ]:
# save topic model 
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# save the topic and document information before reducing topics
#save the topic info
topic_info = topic_model.get_topic_info()
topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/topic_info_bertje.xlsx', index=False)
#save the topic info
doc_info = topic_model.get_document_info(sentences)
doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/doc_info_bertje.xlsx', index=False)

### 2.3 Reduce the outliers

In [36]:
topic_model = loaded_topic_model

In [15]:
# Reduce outliers
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 19/19 [00:03<00:00,  5.48it/s]


In [ ]:
# update topics after reducing outliers
topic_model.update_topics(sentences, topics=new_topics)

In [ ]:
# save the model after reducing outliers
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_reduced_outliers", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# INSPECT topics again # Looks good
topic_model.get_topic_info()

In [30]:
# save the topic and doc info after reducing outliers
topic_model = loaded_topic_model_ro
#save the topic info
topic_info = topic_model.get_topic_info()
topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/topic_info_bertje_reduced.xlsx', index=False)
#save the topic info
doc_info = topic_model.get_document_info(sentences)
doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/doc_info_bertje_reduced.xlsx', index=False)

## 3. Visualize the results

In [150]:
topic_model = loaded_topic_model # select the topic model you want to visualize

In [128]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

### 3.1 Visualize the hierarchy

In [223]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 338/338 [00:04<00:00, 82.88it/s]


### 3.2 Visualize the inter-topic distance map

In [ ]:
# Visualize the inter-topic distance map
topic_model.visualize_topics()

## 4. Merging topics with hierarchy

### 4.1 Merge topics with threshold before outlier-reduction

In [221]:
topic_model = loaded_topic_model # use the model before outlier reduction

In [115]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18991,-1_afspraak_goed_medicatie_ontlasting,"[afspraak, goed, medicatie, ontlasting, vraag, gaan, bloed, mogelijk, krijg, krijgen]",NaN
1,0,1820,0_toilet_buikpijn_pijn_buik,"[toilet, buikpijn, pijn, buik, gevoel, last, krampen, misselijk, diarree, vaker]",NaN
2,1,694,1_sturen_wilt_recept apotheek_recept sturen,"[sturen, wilt, recept apotheek, recept sturen, apotheek sturen, nakijken, vertellen, recept, bel...",NaN
3,2,478,2_nieuw recept_nieuw_recept_graag nieuw,"[nieuw recept, nieuw, recept, graag nieuw, herhaalrecept, pentasa, nethol, puri nethol, puri, gr...",NaN
4,3,443,3_ziekenhuis_apotheek ziekenhuis_ziekenhuis apotheek_ziekenhuis recept,"[ziekenhuis, apotheek ziekenhuis, ziekenhuis apotheek, ziekenhuis recept, prikken ziekenhuis, ap...",NaN
...,...,...,...,...,...
335,334,20,334_geven buikpijn_nachts wakker_extreem_fit moe,"[geven buikpijn, nachts wakker, extreem, fit moe, darmen nachts, flinke hoofdpijn, diarree voel,...",NaN
336,335,20,335_aantal last_voet_lage rug_stijfheid pijn,"[aantal last, voet, lage rug, stijfheid pijn, steken darmen, rechtervoet, pijn stuitje, last ont...",NaN
337,336,20,336_reactie alvast_alvast bedankt_bedankt reactie_alvast,"[reactie alvast, alvast bedankt, bedankt reactie, alvast, antwoord alvast, advies alvast, bedank...",NaN
338,337,20,337_test gedaan_calprotectine test_calprotectine_calpro,"[test gedaan, calprotectine test, calprotectine, calpro, calprotectine ingeleverd, infuus huisar...",NaN


In [183]:
hierarchical_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
337,676,bedankt_hoor_dank_hoor graag_graag,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2...",673,ziekenhuis_apotheek_ontlasting_bloed_last,675,bedankt_graag hoor_hoor graag_alvast_dank,2.852601
336,675,bedankt_graag hoor_hoor graag_alvast_dank,"[4, 35, 44, 46, 58, 69, 76, 82, 91, 93, 98, 99, 103, 117, 119, 128, 138, 143, 145, 150, 166, 177...",627,graag hoor_hoor graag_hoor_hoor hoor_graag,674,bedankt_dank_snelle_reactie_alvast,2.598644
335,674,bedankt_dank_snelle_reactie_alvast,"[4, 44, 46, 58, 69, 76, 82, 91, 93, 98, 99, 117, 128, 138, 145, 166, 177, 185, 199, 200, 212, 21...",636,bericht dank_dank bericht_dank_reactie dank_dank reactie,672,bedankt_alvast_reactie bedankt_snelle_reactie,1.938777
334,673,ziekenhuis_apotheek_ontlasting_bloed_last,"[0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, ...",668,ziekenhuis_afspraak_laten prikken_bloed laten_ingeleverd,671,last_pijn_buikpijn_toilet_recept,1.934506
333,672,bedankt_alvast_reactie bedankt_snelle_reactie,"[4, 44, 46, 58, 69, 76, 82, 91, 93, 98, 99, 138, 145, 166, 177, 199, 200, 212, 214, 219, 233, 24...",657,alvast_alvast bedankt_bedankt alvast_reactie alvast_alvast hartelijk,670,bedankt_reactie bedankt_snelle_fijne_bedankt snelle,1.809557
...,...,...,...,...,...,...,...,...
4,343,reactie bedankt_bedankt reactie_reactie_bedankt_bedankt snelle,"[219, 274]",219,reactie bedankt_bedankt reactie_reactie_bedankt_,274,reactie bedankt_bedankt reactie_reactie_bedankt snelle_bedankt,0.095633
3,342,graag hoor_hoor graag_hoor_graag_bellen hoor,"[35, 143, 184, 338]",341,graag hoor_hoor graag_hoor_graag_graag terug,184,graag hoor_hoor graag_hoor_graag_graag hoort,0.084040
2,341,graag hoor_hoor graag_hoor_graag_graag terug,"[35, 143, 338]",339,graag hoor_hoor graag_hoor_graag_graag terug,143,graag hoor_hoor graag_hoor_graag_hierover hoor,0.047253
1,340,voorbaat_voorbaat dank_dank voorbaat_dank_hartelijk dank,"[74, 77]",77,dank voorbaat_voorbaat dank_voorbaat_dank_,74,voorbaat_voorbaat dank_dank voorbaat_dank_hartelijk dank,0.027227


#### The first merge with a distance threshold

In [224]:
hierarchical_topics[hierarchical_topics["Distance"] <= 0.75]["Topics"].to_list() # change the threshold to see the list of topics with a <0.75 distance with each other

[[4, 93, 166, 219, 274],
 [50, 86],
 [26, 221],
 [191, 235],
 [82, 177, 307],
 [15, 112],
 [228, 299],
 [7, 29, 68],
 [87, 110, 123],
 [89, 337],
 [55, 127],
 [218, 240],
 [84, 253],
 [2, 208],
 [214, 305, 314],
 [114, 202],
 [10, 174],
 [237, 308],
 [11, 33],
 [29, 68],
 [76, 331],
 [88, 160],
 [87, 110],
 [271, 275],
 [99, 233],
 [173, 186],
 [19, 23],
 [90, 256],
 [12, 167],
 [138, 145, 333],
 [27, 97],
 [0, 6],
 [96, 206],
 [177, 307],
 [4, 93, 166],
 [165, 211],
 [164, 292],
 [189, 234],
 [214, 305],
 [247, 298],
 [117, 217],
 [185, 241, 267, 319],
 [69, 200],
 [44, 98],
 [93, 166],
 [35, 119, 143, 150, 184, 279, 338],
 [91, 330, 336],
 [138, 145],
 [35, 143, 150, 184, 279, 338],
 [185, 241, 267],
 [330, 336],
 [35, 143, 184, 279, 338],
 [185, 267],
 [219, 274],
 [35, 143, 184, 338],
 [35, 143, 338],
 [74, 77],
 [35, 338]]

In [225]:
topics_to_merge = hierarchical_topics[hierarchical_topics["Distance"] <= 0.75]["Topics"].to_list() # we will merge the topics with distance < 0.75 (observed from the graph)

topic_model.merge_topics(sentences, topics_to_merge)   # now we merge the topics that are distanced <= 0.9 with each other

In [226]:
# visualize the hierarchy again after the first merge
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics) # we see 285 topics left

100%|██████████| 285/285 [00:03<00:00, 92.68it/s]


#### The second merge with manually grouping clusters
To preserve the details of smaller clusters, we now observe the hierarchy and manually group clusters that are close to each other on the hierarchy and/or have highly similar topic representations

In [232]:
pd.set_option("display.max_colwidth", None)
hierarchical_topics[hierarchical_topics["Parent_Name"].astype(str).str.contains("uur")] # search the parent group you want on the hierarchy here, and copy paste its "Topics"

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
263,549,afspraak_uur_afspraak staan_gepland_afspraak afspraak,"[10, 15, 20, 25, 29, 33, 46, 69, 76, 120, 122, 199, 224, 262, 282]",529,mogelijk_mogelijk afspraak_ophalen_plannen_afspraak,444,afspraak_uur_afspraak staan_coloscopie_gepland,1.214892
211,497,muur_rekken_veranderingen ziektebeeld_ziektebeeld verstandig_ziektebeeld,"[148, 179, 184, 185, 188, 201, 247, 270]",479,muur_kastje_muur gestuurd_kastje muur_rijen,482,ziektebeeld verstandig_veranderingen ziektebeeld_verstandig afspraak_verzetten veranderingen_ziektebeeld,1.008825
201,487,geval_gebeurd_helaas geval_meerdere_spreekuur,"[168, 207, 208, 213, 261]",429,geval_gebeurd_helaas geval_helaas gebeurd_verwarring,476,meerdere_spreekuur_operatiedatum_geprobeerd_telefoons geprobeerd,0.992867
193,479,muur_kastje_muur gestuurd_kastje muur_rijen,"[148, 184, 188, 201]",430,muur_muur gestuurd_kastje muur_kastje_checks,384,rijen_aandoening_begeleider_engels_ziekteverloop,0.987596
191,477,gebruik_foliumzuur_psylliumvezels_daags_graag medicatie,"[74, 75, 111, 115, 248]",350,gebruik_daags_maal daags_slik_maal,319,foliumzuur_psylliumvezels_graag medicatie_mtx_medicatie bestellen,0.986108
190,476,meerdere_spreekuur_operatiedatum_geprobeerd_telefoons geprobeerd,"[207, 208, 213]",423,meerdere_operatiedatum_geprobeerd_telefoons geprobeerd_telefoons,207,spreekuur_huisartsenpost_werkdag_chromo ingepland_staat colonosc,0.985494
187,473,apotheek apotheek_gefaxt_apotheek_gefaxt apotheek_opgestuurd,"[63, 86, 139, 144]",351,apotheek apotheek_apotheek_adres_nieuwe adres_service apotheek,387,gefaxt apotheek_opgestuurd_gefaxt_apotheek herhaalrecept_belt,0.985215
158,444,afspraak_uur_afspraak staan_coloscopie_gepland,"[10, 15, 25, 46, 199]",342,afspraak_uur_afspraak staan_coloscopie_staan,199,goedendag_hoor afspraak_urologie_afspraak plaatsvind_goed prima,0.955104
156,442,duurde_duurde langer_werking medicijn_keer duurde_opperde,"[185, 270]",270,keer duurde_komt terug_duurde_ontsteking helemaal_duurde langer,185,opperde_werking medicijn_rekken_problemen geven_gezegd,0.953441
144,430,muur_muur gestuurd_kastje muur_kastje_checks,"[184, 188]",188,kastje muur_kastje_muur gestuurd_muur_gestuurd,184,streepje_stap_veronderstelling_chirurgie_schema,0.943550


In [ ]:
topics_to_merge = [
[6, 7, 28, 30, 38, 39, 42, 45, 54, 60, 65, 68, 70, 81, 93, 99, 106, 110, 129, 130, 172, 187, 194, 212, 233, 240], #bedankt_graag hoor_hoor graag_hoor_alvast
[1, 2, 57], #recept_nieuw recept_nieuw_sturen_apotheek	
[63, 86, 139, 144], #apotheek apotheek_gefaxt_apotheek_gefaxt apotheek_opgestuurd	
[11, 24, 78, 80, 125, 134, 143], #uitslag_bekend uitslag_uitslagen_bekend_uitslag ontlasting	
[20, 29, 33], #mogelijk_mogelijk afspraak_plannen_afspraak plannen_afspraak	
[10, 25], #uur_afspraak_afspraak staan_afspraak afspraak_staan	

] 



### 3.2 Manual merge

In [51]:
# merge some topics
topics_to_merge = [[20, 328, 207, 301], # Diet
                    [281, 158, 278, 263, 259, 41, 13, 220], # Consultation/Hospital Visit
                    [181, 276, 243, 175, 163, 112,311, 246, 325, 216, 312, 321, 303, 306, 175, 163, 112, 188, 40, 129, 142, 204, 148], # Health Update
                    [261, 297, 190, 53, 70, 131, 171, 251, 323, 203, 225, 300, 160, 67, 111, 252, 213, 176, 258, 257, 66, 224, 162, 52, 49,
                    38, 153, 226, 152, 168, 309, 180, 104, 196, 201], # Medical Advice/Discussion
                    [192, 324, 54, 110], # Exam
                    [210, 103, 135, 106, 63, 62, 288, 113, 149, 81, 109, 92, 159, 14, 88, 332, 108, 79, 95, 134, 250, 75, 284, 169, 266, 193,
                    51, 37, 194, 72], # Medication
                    [329, 268, 215], # Pregnancy
                    [147, 86, 50, 27, 97, 260, 61, 71, 47, 326, 209, 273, 335, 182, 32, 0, 6, 105, 286, 334, 126, 227, 15, 231, 313, 293, 78, 146, 320, 270, 255, 42], # Symptoms
                    [], # Surgery
                    [197, 244, 136, 229, 154, 329, 43, 248, 302, 172], # Vaccination
                    [276, 243], # Work
                    [56, 141, 235, 191, 232, 36, 33, 265, 285, 57, 264, 179, 205, 48, 115, 94, 107, 118, 64,
                    291, 295, 236, 254, 285, 80, 22, 280, 242, 289, 59, 223, 187, 294, 282, 73, 240, 161, 25, 198, 310, 178, 316, 26, 221, 157,
                    55, 127, 24, 17, 16, 34, 5], # Admin comm (survey, patient info, documents)
                    [122, 18, 137, 144, 164, 292, 83, 87, 123, 31, 116, 10, 174, 230], # Appointment/contact
                    [3, 256, 90, 296, 2, 208, 1, 202, 84, 253, 114, 290, 315, 100, 317, 8], # Pharmacy
                    [183, 121, 165, 211, 167, 12, 21, 132, 65, 337, 89, 9, 125, 60, 28, 23, 19, 29, 68, 7, 173, 186, 304, 124], # Test
                    [239, 151, 277, 287, 222, 262, 139, 102, 85, 322, 189, 150, 234, 119, 143, 35, 338, 184, 279, 98, 44, 233, 99, 212, 331, 76, 199, 298, 247, 128,
                    336, 91, 330, 333, 138, 145, 245, 46, 58, 314, 214, 305, 166, 93, 4, 217, 117, 274, 219, 200, 69, 319, 241, 185, 267, 74, 77,
                    249, 82, 307, 177, 318, 269, 156, 195, 238, 218, 272, 155, 275, 271, 299, 228, 308, 237, 96, 206, 133, 101, 327, 170, 39, 130] # Other

]
topic_model.merge_topics(sentences, topics_to_merge)  

In [ ]:
doc_info = topic_model.get_document_info(sentences)
doc_info[doc_info["Topic"] == 301]

In [ ]:

# merge some topics
topics_to_merge = [[2, 12]
]
topic_model.merge_topics(sentences, topics_to_merge)  

In [ ]:
# Manually Put Labels in the Label Column
# Create the customized label column
topic_info["Label"] = None

In [ ]:
topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/final_topic_info_bertje.xlsx', index=False)

In [ ]:
labeled_topic_info = pd.read_excel('/workspace/mijnidbcoachnlp/data/result_data/merged_topic_info_bertje_labeled.xlsx')

In [ ]:
labeled_topic_info.rename(columns={"Unnamed: 5": "Model_Label"}, inplace=True)

In [ ]:
merged_doc_info = topic_model.get_document_info(sentences)
merged_doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/merged_doc_info_bertje.xlsx', index=False)

In [ ]:
merged_doc_info["Sentence_ID"] = merged_doc_info.index


In [ ]:
annotated_df['Sentence_ID'] = annotated_df['Sentence_ID'].astype(str)
merged_doc_info['Sentence_ID'] = merged_doc_info['Sentence_ID'].astype(str)

In [ ]:
# Now perform the merge
annotated_df = pd.merge(
    annotated_df,
    merged_doc_info[['Sentence_ID', 'Topic', 'Top_n_words']],
    on='Sentence_ID',
    how='left'
)

In [ ]:
annotated_df = pd.merge(
    annotated_df,
    labeled_topic_info[['Topic', 'Model_Label']],
    on='Topic',
    how='left'
)

In [ ]:
# Filter the DataFrame for manual labels equal to "A4"
label = "M7"
test_df = annotated_df[annotated_df["Manual_Label"] == label]

# Calculate the accuracy as the percentage of matches
accuracy = (test_df['Manual_Label'] == test_df['Model_Label']).sum() / len(test_df)

# Output the accuracy
print(f"Accuracy for Manual_Label == {label}: {accuracy * 100:.2f}%")
